# Sensibilidad de MARGIN

Este notebook evalua el impacto de distintos percentiles de MARGIN en el desempeno de la U-Net.
Cada MARGIN se entrena desde cero y se evalua con las metricas vectoriales.

In [1]:
import torch, random, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from shapely import wkt
from shapely.geometry import shape
from shapely.ops import unary_union
from shapely.validation import make_valid
from affine import Affine
import rasterio.features
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

def set_seed(seed=42, deterministic=True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

set_seed(42, deterministic=True)
DEVICE = get_device()
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("device:", DEVICE)

device: mps


In [2]:
DATA_PATH = "Datos Procesados/final/transitions_master.parquet"
df = pd.read_parquet(DATA_PATH)
train = df[df["split"] == "train"].copy()
val   = df[df["split"] == "val"].copy()
test  = df[df["split"] == "test"].copy()
print(f"Transiciones: {len(df)} | train={len(train)} val={len(val)} test={len(test)}")

Transiciones: 1171 | train=825 val=159 test=187


In [3]:
FEATURE_COLS_20 = [
    "t_area_ha", "t_perimeter_m", "t_compactness", "t_n_vertices",
    "t_bbox_width_m", "t_bbox_height_m",
    "t_srtm_elev_mean_m", "t_srtm_elev_std_m",
    "t_srtm_slope_mean_deg", "t_srtm_slope_max_deg",
    "t_srtm_aspect_sin_mean", "t_srtm_aspect_cos_mean",
    "t_era5_fire_mean_temperature_2m", "t_era5_fire_mean_relative_humidity_2m",
    "t_era5_fire_mean_wind_speed_10m",
    "t_era5_fire_wind_dir_sin_mean", "t_era5_fire_wind_dir_cos_mean",
    "t_era5_fire_dryness_index",
    "t_firms_total_n_10km", "t_firms_viirs_frp_mean_10km",
]

feat_imputer = SimpleImputer(strategy="median")
feat_scaler  = StandardScaler()

Xf_train = feat_imputer.fit_transform(train[FEATURE_COLS_20]).astype(np.float32)
Xf_val   = feat_imputer.transform(val[FEATURE_COLS_20]).astype(np.float32)
Xf_test  = feat_imputer.transform(test[FEATURE_COLS_20]).astype(np.float32)

Xf_train = feat_scaler.fit_transform(Xf_train).astype(np.float32)
Xf_val   = feat_scaler.transform(Xf_val).astype(np.float32)
Xf_test  = feat_scaler.transform(Xf_test).astype(np.float32)

In [4]:
RASTER_SIZE = 128
MAX_EPOCHS  = 50   # igual que modelo_final.ipynb para resultados comparables
PATIENCE    = 10   # igual que modelo_final.ipynb
BATCH_SIZE  = 16
THRESHOLDS  = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
MARGIN_PCTS = [0.8, 0.83, 0.85, 0.87, 0.90, 0.93, 0.95, 0.97, 0.99]
print("Margen pcts:", MARGIN_PCTS)
print(f"MAX_EPOCHS={MAX_EPOCHS}, PATIENCE={PATIENCE} — idénticos a modelo_final.ipynb")

Margen pcts: [0.8, 0.83, 0.85, 0.87, 0.9, 0.93, 0.95, 0.97, 0.99]
MAX_EPOCHS=50, PATIENCE=10 — idénticos a modelo_final.ipynb


In [5]:
def polygonal_only(geom):
    if geom is None or geom.is_empty:
        return None
    geom = make_valid(geom)
    if geom.geom_type in ("Polygon", "MultiPolygon"):
        return geom
    if geom.geom_type == "GeometryCollection":
        parts = [g for g in geom.geoms
                 if g.geom_type in ("Polygon", "MultiPolygon") and not g.is_empty]
        return unary_union(parts) if parts else None
    return None

def poly_iou(a, b):
    if a is None or b is None or a.is_empty or b.is_empty:
        return 0.0
    inter = a.intersection(b).area
    union = a.union(b).area
    return inter / union if union > 0 else 0.0

def poly_dice(a, b):
    if a is None or b is None or a.is_empty or b.is_empty:
        return 0.0
    inter = a.intersection(b).area
    denom = a.area + b.area
    return 2 * inter / denom if denom > 0 else 0.0

In [6]:
def required_margin_factor(P_t, P_t1):
    minx, miny, maxx, maxy = P_t.bounds
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    base_side = max(maxx - minx, maxy - miny, 1.0)
    a, b, c, d = P_t1.bounds
    required_half = max(abs(a - cx), abs(c - cx), abs(b - cy), abs(d - cy))
    return max(1.0, 2 * required_half / base_side)

margin_factors = [
    required_margin_factor(
        wkt.loads(row["geometry_t_wkt"]),
        wkt.loads(row["geometry_t1_wkt"])
    )
    for _, row in train.iterrows()
]

margin_candidates = [float(np.quantile(margin_factors, p)) for p in MARGIN_PCTS]
print("MARGIN candidates:", margin_candidates)

MARGIN candidates: [13.74737490588475, 15.20815415696483, 16.034611830898864, 16.780988503043563, 19.008824827794474, 22.181709928560906, 25.460698811325607, 32.60953700586031, 65.79576590469485]


In [7]:
def make_window(P_t, size, margin):
    minx, miny, maxx, maxy = P_t.bounds
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    side = max(maxx - minx, maxy - miny, 1.0) * margin
    half = side / 2
    bx0, by0 = cx - half, cy - half
    bx1, by1 = cx + half, cy + half
    px = side / size
    transform = Affine(px, 0, bx0, 0, -px, by1)
    return transform, (bx0, by0, bx1, by1)

def rasterize_poly(poly, transform, size):
    return rasterio.features.rasterize(
        [(poly, 1)],
        out_shape=(size, size),
        transform=transform,
        fill=0,
        dtype=np.uint8,
        all_touched=True,
    ).astype(np.float32)

class FireDataset(Dataset):
    def __init__(self, df_split, feat_array, size, margin):
        self.df = df_split.reset_index(drop=True)
        self.feat = feat_array.astype(np.float32)
        self.size = size
        self.margin = margin

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        P_t  = wkt.loads(row["geometry_t_wkt"])
        P_t1 = wkt.loads(row["geometry_t1_wkt"])
        transform, _ = make_window(P_t, self.size, self.margin)
        mask_t  = rasterize_poly(P_t,  transform, self.size)
        mask_t1 = rasterize_poly(P_t1, transform, self.size)
        f = self.feat[idx]
        f_maps = np.broadcast_to(f[:, None, None], (len(f), self.size, self.size)).copy()
        x = np.concatenate([mask_t[None], f_maps], axis=0)
        y = mask_t1[None]
        return (
            torch.from_numpy(x),
            torch.from_numpy(y),
            {
                "transform": np.array(transform.to_gdal()),
                "geom_t": P_t,
                "geom_t1": P_t1,
            },
        )

def collate_fn(batch):
    xs   = torch.stack([b[0] for b in batch])
    ys   = torch.stack([b[1] for b in batch])
    meta = [b[2] for b in batch]
    return xs, ys, meta

In [8]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNetSmall(nn.Module):
    def __init__(self, in_ch=21, base=32):
        super().__init__()
        b = base
        self.d1, self.p1 = DoubleConv(in_ch, b), nn.MaxPool2d(2)
        self.d2, self.p2 = DoubleConv(b, b*2), nn.MaxPool2d(2)
        self.d3, self.p3 = DoubleConv(b*2, b*4), nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(b*4, b*8)
        self.u3 = nn.ConvTranspose2d(b*8, b*4, 2, stride=2)
        self.c3 = DoubleConv(b*8, b*4)
        self.u2 = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.c2 = DoubleConv(b*4, b*2)
        self.u1 = nn.ConvTranspose2d(b*2, b, 2, stride=2)
        self.c1 = DoubleConv(b*2, b)
        self.out = nn.Conv2d(b, 1, 1)
    def forward(self, x):
        x1 = self.d1(x)
        x2 = self.d2(self.p1(x1))
        x3 = self.d3(self.p2(x2))
        xb = self.bottleneck(self.p3(x3))
        y = self.c3(torch.cat([self.u3(xb), x3], dim=1))
        y = self.c2(torch.cat([self.u2(y), x2], dim=1))
        y = self.c1(torch.cat([self.u1(y), x1], dim=1))
        return self.out(y)

def dice_loss(logits, target, eps=1e-7):
    p = torch.sigmoid(logits).view(logits.size(0), -1)
    t = target.view(target.size(0), -1)
    inter = (p * t).sum(1)
    denom = p.sum(1) + t.sum(1)
    return 1 - (2 * inter + eps) / (denom + eps)

def combined_loss(logits, target):
    bce  = F.binary_cross_entropy_with_logits(logits, target)
    dice = dice_loss(logits, target).mean()
    return 0.6 * bce + 0.4 * dice

def mask_iou_batch(logits, target, tau=0.5):
    pred  = (torch.sigmoid(logits) >= tau).float()
    inter = (pred * target).sum((1, 2, 3))
    union = pred.sum((1, 2, 3)) + target.sum((1, 2, 3)) - inter
    return ((inter + 1e-6) / (union + 1e-6)).mean().item()

In [9]:
def extended_metrics(loader, model, tau, split_name):
    model.eval()
    poly_ious_, poly_dices_ = [], []
    area_errs, centroid_disps, perim_errs = [], [], []
    n_valid = 0
    n_total = 0
    with torch.no_grad():
        for x, y, meta in loader:
            logits = model(x.to(DEVICE))
            probs  = torch.sigmoid(logits).cpu().numpy()
            y_np   = y.numpy()
            for i in range(x.shape[0]):
                n_total += 1
                pred_mask = (probs[i, 0] >= tau).astype(np.uint8)
                transform = Affine.from_gdal(*meta[i]["transform"])
                polys = [
                    shape(g)
                    for g, v in rasterio.features.shapes(
                        pred_mask, mask=pred_mask, transform=transform
                    )
                    if int(v) == 1
                ]
                true_poly = polygonal_only(meta[i]["geom_t1"])
                if true_poly is None:
                    poly_ious_.append(0.0)
                    poly_dices_.append(0.0)
                    area_errs.append(np.nan)
                    centroid_disps.append(np.nan)
                    perim_errs.append(np.nan)
                    continue
                if not polys:
                    poly_ious_.append(0.0)
                    poly_dices_.append(0.0)
                    area_errs.append(true_poly.area / 1e4)
                    centroid_disps.append(np.nan)
                    perim_errs.append(true_poly.length)
                    continue
                pred_poly = polygonal_only(unary_union(polys))
                if pred_poly is None:
                    poly_ious_.append(0.0)
                    poly_dices_.append(0.0)
                    area_errs.append(true_poly.area / 1e4)
                    centroid_disps.append(np.nan)
                    perim_errs.append(true_poly.length)
                    continue
                n_valid += 1
                poly_ious_.append(poly_iou(pred_poly, true_poly))
                poly_dices_.append(poly_dice(pred_poly, true_poly))
                area_errs.append(abs(pred_poly.area - true_poly.area) / 1e4)
                centroid_disps.append(pred_poly.centroid.distance(true_poly.centroid))
                perim_errs.append(abs(pred_poly.length - true_poly.length))

    iou_arr  = np.array(poly_ious_)
    area_arr = np.array(area_errs,      dtype=float)
    cent_arr = np.array(centroid_disps, dtype=float)
    peri_arr = np.array(perim_errs,     dtype=float)

    return {
        "split":                   split_name,
        "poly_iou":                float(np.mean(iou_arr))          if len(iou_arr)  else None,
        "poly_iou_median":         float(np.median(iou_arr))        if len(iou_arr)  else None,
        "poly_dice":               float(np.mean(np.array(poly_dices_))) if poly_dices_ else None,
        "area_err_ha":             float(np.nanmean(area_arr))      if area_errs     else None,
        "area_err_ha_median":      float(np.nanmedian(area_arr))    if area_errs     else None,
        "centroid_disp_m":         float(np.nanmean(cent_arr))      if centroid_disps else None,
        "centroid_disp_m_median":  float(np.nanmedian(cent_arr))    if centroid_disps else None,
        "perim_err_m":             float(np.nanmean(peri_arr))      if perim_errs    else None,
        "perim_err_m_median":      float(np.nanmedian(peri_arr))    if perim_errs    else None,
        "pct_nonempty_pred_geom":  n_valid / n_total if n_total else 0.0,
        "n_total":                 n_total,
    }

In [10]:
def train_and_eval_for_margin(margin, tag):
    train_loader = DataLoader(
        FireDataset(train, Xf_train, RASTER_SIZE, margin),
        batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0
    )
    val_loader = DataLoader(
        FireDataset(val, Xf_val, RASTER_SIZE, margin),
        batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0
    )
    test_loader = DataLoader(
        FireDataset(test, Xf_test, RASTER_SIZE, margin),
        batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0
    )

    model = UNetSmall(in_ch=1 + len(FEATURE_COLS_20), base=32).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    best_val_iou = -1.0
    wait = 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses = []
        for x, y, _ in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = combined_loss(model(x), y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        model.eval()
        val_ious = []
        with torch.no_grad():
            for x, y, _ in val_loader:
                val_ious.append(mask_iou_batch(model(x.to(DEVICE)), y.to(DEVICE)))
        val_iou = float(np.mean(val_ious))
        if val_iou > best_val_iou:
            best_val_iou = val_iou
            wait = 0
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state)

    tau_metrics = {}
    for tau in THRESHOLDS:
        m = extended_metrics(val_loader, model, tau, "val")
        tau_metrics[tau] = m
    tau_star = max(tau_metrics, key=lambda t: tau_metrics[t]["poly_iou"])

    val_metrics  = dict(tau_metrics[tau_star])
    test_metrics = extended_metrics(test_loader, model, tau_star, "test")

    return {
        "tag":    tag,
        "margin": float(margin),
        "tau_star": float(tau_star),
        # --- validación ---
        "val_poly_iou":               val_metrics["poly_iou"],
        "val_poly_iou_median":        val_metrics["poly_iou_median"],
        "val_poly_dice":              val_metrics["poly_dice"],
        "val_area_err_ha":            val_metrics["area_err_ha"],
        "val_area_err_ha_median":     val_metrics["area_err_ha_median"],
        "val_centroid_disp_m":        val_metrics["centroid_disp_m"],
        "val_centroid_disp_m_median": val_metrics["centroid_disp_m_median"],
        "val_perim_err_m":            val_metrics["perim_err_m"],
        "val_perim_err_m_median":     val_metrics["perim_err_m_median"],
        # --- prueba ---
        "test_poly_iou":               test_metrics["poly_iou"],
        "test_poly_iou_median":        test_metrics["poly_iou_median"],
        "test_poly_dice":              test_metrics["poly_dice"],
        "test_area_err_ha":            test_metrics["area_err_ha"],
        "test_area_err_ha_median":     test_metrics["area_err_ha_median"],
        "test_centroid_disp_m":        test_metrics["centroid_disp_m"],
        "test_centroid_disp_m_median": test_metrics["centroid_disp_m_median"],
        "test_perim_err_m":            test_metrics["perim_err_m"],
        "test_perim_err_m_median":     test_metrics["perim_err_m_median"],
        "test_pct_nonempty_geom":      test_metrics["pct_nonempty_pred_geom"],
    }

In [11]:
results = []
for p, m in zip(MARGIN_PCTS, margin_candidates):
    print(f"\n=== MARGIN P{int(p*100)} = {m:.3f} ===")
    res = train_and_eval_for_margin(m, f"P{int(p*100)}")
    results.append(res)

summary = pd.DataFrame(results)
summary.to_csv(OUTPUT_DIR / "margin_sensitivity.csv", index=False)
summary


=== MARGIN P80 = 13.747 ===

=== MARGIN P83 = 15.208 ===

=== MARGIN P85 = 16.035 ===

=== MARGIN P87 = 16.781 ===

=== MARGIN P90 = 19.009 ===

=== MARGIN P93 = 22.182 ===

=== MARGIN P95 = 25.461 ===

=== MARGIN P97 = 32.610 ===

=== MARGIN P99 = 65.796 ===


,tag,margin,tau_star,val_poly_iou,val_poly_iou_median,val_poly_dice,val_area_err_ha,val_area_err_ha_median,val_centroid_disp_m,val_centroid_disp_m_median,...,test_poly_iou,test_poly_iou_median,test_poly_dice,test_area_err_ha,test_area_err_ha_median,test_centroid_disp_m,test_centroid_disp_m_median,test_perim_err_m,test_perim_err_m_median,test_pct_nonempty_geom
0,P80,13.747375,0.5,0.445129,0.473862,0.560435,38.566969,16.131068,72.463012,44.641008,...,0.433010,0.487750,0.547490,101.111336,10.811574,87.937794,47.327940,1851.370224,724.196248,1.000000
1,P83,15.208154,0.3,0.466044,0.519719,0.582296,33.726785,16.493953,64.456232,44.682280,...,0.435270,0.487713,0.547974,5372.940846,13.600820,90.293467,48.239076,3436.770855,753.026959,1.000000
2,P85,16.034612,0.6,0.498251,0.536872,0.628032,37.694742,9.409194,61.377660,38.121945,...,0.459579,0.508698,0.588837,48.668706,8.843990,85.545791,52.568221,1770.000279,413.419669,1.000000
3,P87,16.780989,0.7,0.440226,0.466435,0.560166,81.568827,13.780745,62.682696,40.691562,...,0.432704,0.510072,0.548983,1072.458833,11.460927,88.864430,55.009164,2977.232988,689.851918,1.000000
4,P90,19.008825,0.7,0.443068,0.490061,0.560972,42.642589,17.076590,67.959569,47.285675,...,0.433204,0.515987,0.549579,7103.857834,14.707503,92.531701,48.563785,6329.433134,901.640774,1.000000
5,P93,22.181710,0.5,0.428687,0.433173,0.551654,52.641533,16.267646,67.019296,45.038019,...,0.426637,0.486243,0.544946,64.251262,13.373669,92.683338,46.992258,2340.566767,747.937645,1.000000
6,P95,25.460699,0.6,0.430982,0.452132,0.550254,40.656596,14.057171,70.225437,48.346621,...,0.431120,0.501417,0.547512,57.197825,12.432169,91.945970,50.712861,1973.743568,569.099223,1.000000
7,P97,32.609537,0.6,0.475973,0.500930,0.610372,50.875008,9.671763,69.492183,48.275621,...,0.442249,0.478990,0.574363,106.578646,9.051120,96.430646,55.543583,1679.454952,484.408323,1.000000
8,P99,65.795766,0.6,0.342155,0.320878,0.471586,158.047177,20.677280,77.824592,47.775188,...,0.348200,0.349651,0.473114,191.667993,16.178787,95.189996,51.750081,2565.405857,1012.600629,0.994652
